In [1]:
import pandas as pd
import seaborn as sns

taxis = sns.load_dataset("taxis")

# Conversion en type datetime
taxis["pickup"] = pd.to_datetime(taxis["pickup"])
taxis["dropoff"] = pd.to_datetime(taxis["dropoff"])

print(taxis.shape)  # Doit afficher (6433, 14)
print("Période :", taxis["pickup"].min(), "à", taxis["pickup"].max())

(6433, 14)
Période : 2019-02-28 23:29:03 à 2019-03-31 23:43:45


In [2]:
# Création d'une colonne avec la date seule (jour)
taxis["date"] = taxis["pickup"].dt.date

# Nombre de courses (count) et chiffre d'affaires (sum sur 'total')
bilan_quotidien = taxis.groupby("date").agg(
    nb_courses=("total", "count"), chiffre_affaires=("total", "sum")
)
print(bilan_quotidien.head())

            nb_courses  chiffre_affaires
date                                    
2019-02-28           1              6.30
2019-03-01         241           4213.83
2019-03-02         198           3319.02
2019-03-03         169           3027.32
2019-03-04         171           3269.08


In [3]:
taxis["heure"] = taxis["pickup"].dt.hour
courses_par_heure = taxis["heure"].value_counts().sort_index()

heure_de_pointe = taxis["heure"].value_counts().idxmax()
print("Heure la plus chargée :", heure_de_pointe, "h")

Heure la plus chargée : 18 h


In [5]:
taxis["jour_semaine"] = taxis["pickup"].dt.day_name()
pourboire_moyen_jour = taxis.groupby("jour_semaine")["tip"].mean().sort_values(
    ascending=False
)
print(pourboire_moyen_jour)

jour_semaine
Wednesday    2.087329
Thursday     2.062177
Monday       2.027316
Tuesday      2.001370
Friday       1.980108
Sunday       1.908710
Saturday     1.815143
Name: tip, dtype: float64


In [6]:
# Calcul de la durée en minutes
taxis["duree_min"] = (taxis["dropoff"] - taxis["pickup"]).dt.total_seconds() / 60

# Repérage des durées aberrantes (ex: <= 0 min ou > 180 min)
durees_aberrantes = taxis[(taxis["duree_min"] <= 0) | (taxis["duree_min"] > 180)]
print(f"Nombre de durées aberrantes : {len(durees_aberrantes)}")

# Nettoyage : conservation des courses réalistes (ex: entre 1 min et 180 min)
taxis_propre = taxis[(taxis["duree_min"] > 0) & (taxis["duree_min"] <= 180)]

Nombre de durées aberrantes : 6
